# 3D Reconstruction Pipeline
**SfM → MVS → 3DGS 완전 자동화**

## 시작 전 체크리스트
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- Google Drive에 `recon3d/images/` 폴더 생성 후 사진 업로드
- 사진 권장: 30장 이상, 인접 사진 간 60% 이상 겹침

---
## CELL 1 | GPU 및 CUDA 환경 확인 (반드시 먼저 실행)

In [ ]:
import torch

print('='*50)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('='*50)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {vram:.1f} GB')
else:
    print('CUDA 불가 -- 런타임을 T4 GPU로 변경하세요!')

Tesla T4, 15360 MiB
PyTorch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
VRAM: 14.6 GB


---
## CELL 2 | Google Drive 마운트 및 경로 설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_NAME = 'recon3d'
BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
IMAGES = f'{BASE}/images'
SPARSE = f'{BASE}/sparse'
DENSE  = f'{BASE}/dense'
OUTPUT = f'{BASE}/output'
DB     = f'{BASE}/database.db'

Mounted at /content/drive


In [2]:
%%bash
git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive --quiet
cd gaussian-splatting
pip install submodules/diff-gaussian-rasterization -q
pip install submodules/simple-knn -q
pip install plyfile tqdm pillow -q
echo '설치 완료!'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.2 MB/s eta 0:00:00
설치 완료!


In [4]:
# train.py의 torch.load 부분을 수정
!sed -i 's/torch.load(checkpoint)/torch.load(checkpoint, weights_only=False)/g' \
    /content/gaussian-splatting/train.py

print('수정 완료! 확인:')
!grep 'torch.load' /content/gaussian-splatting/train.py

수정 완료! 확인:
        (model_params, first_iter) = torch.load(checkpoint, weights_only=False)


In [5]:
import os, subprocess

PROJECT_NAME = 'recon3d'
BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
GS_OUT = f'{BASE}/output/gs_result'

env = os.environ.copy()
env['QT_QPA_PLATFORM'] = 'offscreen'

print('20000에서 이어받기...\n')

p = subprocess.Popen(
    ['python', 'train.py',
     '-s', BASE,
     '-m', GS_OUT,
     '--iterations', '30000',
     '--save_iterations', '30000',
     '--checkpoint_iterations', '30000',
     '--test_iterations', '30000',
     '--densify_until_iter', '5000',
     '--start_checkpoint', f'{GS_OUT}/chkpnt20000.pth',
    ],
    cwd='/content/gaussian-splatting',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env
)
for line in p.stdout:
    print(line, end='', flush=True)
p.wait()

ply = f'{GS_OUT}/point_cloud/iteration_30000/point_cloud.ply'
if os.path.exists(ply):
    mb = os.path.getsize(ply) / 1024**2
    print(f'\n✅ 완료! {mb:.1f} MB')

20000에서 이어받기...

2026-05-26 18:28:00.998318: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-26 18:28:01.069981: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/drive/MyDrive/recon3d/output/gs_result
Output folder: /content/drive/MyDrive/recon3d/output/gs_result [26/05 18:28:05]

Reading camera 1/25
Reading camera 2/25
Reading camera 3/25
Reading camera 4/25
Reading camera 5/25
Reading camera 6/25
Reading camera 7/25
Reading camera 8/25
Reading camera 9/25
Reading camera 10/25
Reading camera 1

In [6]:
import numpy as np, struct, os
from plyfile import PlyData

def ply_to_splat(input_ply, output_splat):
    print(f'변환 중: {input_ply}')
    plydata = PlyData.read(input_ply)
    v = plydata['vertex']

    means  = np.stack([v['x'], v['y'], v['z']], axis=1)
    scales = np.exp(np.stack([v['scale_0'], v['scale_1'], v['scale_2']], axis=1))
    rots   = np.stack([v['rot_0'], v['rot_1'], v['rot_2'], v['rot_3']], axis=1)
    rots   = rots / np.linalg.norm(rots, axis=1, keepdims=True)
    opac   = 1 / (1 + np.exp(-v['opacity']))

    SH_C0  = 0.28209479177387814
    colors = np.clip(
        (np.stack([v['f_dc_0'], v['f_dc_1'], v['f_dc_2']], axis=1) * SH_C0 + 0.5) * 255,
        0, 255
    ).astype(np.uint8)

    n = len(means)
    print(f'Gaussian 수: {n:,}개')

    idx = np.argsort(-opac)
    means, scales, rots, opac, colors = (
        means[idx], scales[idx], rots[idx], opac[idx], colors[idx]
    )

    with open(output_splat, 'wb') as f:
        for i in range(n):
            f.write(struct.pack('fff', *means[i]))
            f.write(struct.pack('fff', *scales[i]))
            f.write(struct.pack('BBBB',
                colors[i,0], colors[i,1], colors[i,2],
                int(opac[i] * 255)))
            q = rots[i]
            f.write(struct.pack('BBBB',
                int((q[1]*0.5+0.5)*255),
                int((q[2]*0.5+0.5)*255),
                int((q[3]*0.5+0.5)*255),
                int((q[0]*0.5+0.5)*255)))

    mb = os.path.getsize(output_splat) / 1024**2
    print(f'✅ 저장 완료: {output_splat} ({mb:.1f} MB)')

PROJECT_NAME = 'recon3d'
BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
GS_OUT = f'{BASE}/output/gs_result'

ply_path   = f'{GS_OUT}/point_cloud/iteration_30000/point_cloud.ply'
splat_path = f'{BASE}/output/scene.splat'

ply_to_splat(ply_path, splat_path)
print('\n🌐 WebGL 확인: https://supersplat.com 에 scene.splat 업로드!')

변환 중: /content/drive/MyDrive/recon3d/output/gs_result/point_cloud/iteration_30000/point_cloud.ply
Gaussian 수: 2,758,551개
✅ 저장 완료: /content/drive/MyDrive/recon3d/output/scene.splat (84.2 MB)

🌐 WebGL 확인: https://supersplat.com 에 scene.splat 업로드!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ===== 여기만 수정 =====
PROJECT_NAME = 'recon3d'
# ======================

BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
IMAGES = f'{BASE}/images'
SPARSE = f'{BASE}/sparse'
DENSE  = f'{BASE}/dense'
OUTPUT = f'{BASE}/output'
DB     = f'{BASE}/database.db'

for d in [IMAGES, SPARSE, DENSE, OUTPUT]:
    os.makedirs(d, exist_ok=True)

exts = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
imgs = [f for f in os.listdir(IMAGES) if f.endswith(exts)]
print(f'발견된 이미지: {len(imgs)}장')

if len(imgs) < 15:
    print('WARNING: 이미지가 15장 미만입니다. 30장 이상 권장')
elif len(imgs) < 30:
    print('이미지 수 보통. 30장 이상이 더 좋아요.')
else:
    print('이미지 수 충분!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
발견된 이미지: 25장
이미지 수 보통. 30장 이상이 더 좋아요.


---
## CELL 3 | COLMAP 설치

In [ ]:
%%bash
echo 'COLMAP 설치 중...'
apt-get update -qq
apt-get install -y -qq colmap 2>&1 | tail -3
echo 'COLMAP 버전:'
colmap --version
pip install pycolmap -q
python -c "import pycolmap; print('pycolmap OK')"

COLMAP 설치 중...
Setting up colmap (3.7-2) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
COLMAP 버전:
pycolmap OK


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: Command `--version` not recognized. To list the available commands, run `colmap help`.


---
## CELL 4 | SfM — Feature Extraction
> max_num_features 를 높여서 매칭 실패 방지

In [ ]:
import os, sqlite3, subprocess

PROJECT_NAME = 'recon3d'
BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
IMAGES = f'{BASE}/images'
SPARSE = f'{BASE}/sparse'
DENSE  = f'{BASE}/dense'
OUTPUT = f'{BASE}/output'
DB     = f'{BASE}/database.db'

for d in [IMAGES, SPARSE, DENSE, OUTPUT]:
    os.makedirs(d, exist_ok=True)

if os.path.exists(DB):
    os.remove(DB)
    print('기존 DB 삭제')

print('Feature 추출 중... (CPU 모드, 2~5분 소요)\n')

env = os.environ.copy()
env['QT_QPA_PLATFORM'] = 'offscreen'

cmd = [
    'colmap', 'feature_extractor',
    '--database_path', DB,
    '--image_path', IMAGES,
    '--ImageReader.camera_model', 'SIMPLE_RADIAL',
    '--ImageReader.single_camera', '1',
    '--SiftExtraction.max_num_features', '8192',
    '--SiftExtraction.use_gpu', '0',  # CPU 모드로 변경
    '--SiftExtraction.num_threads', '4',
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\n종료 코드: {process.returncode}')

if process.returncode == 0:
    conn = sqlite3.connect(DB)
    n_img = conn.execute('SELECT COUNT(*) FROM images').fetchone()[0]
    n_kp  = conn.execute('SELECT COUNT(*) FROM keypoints').fetchone()[0]
    conn.close()
    avg = n_kp // max(n_img, 1)
    print(f'이미지: {n_img}장  |  총 keypoint: {n_kp:,}  |  평균: {avg:,}/장')
    if avg < 500:
        print('WARNING: keypoint 적음')
else:
    print('ERROR: 실패')

기존 DB 삭제
Feature 추출 중... (CPU 모드, 2~5분 소요)


Feature extraction

Processed file [1/25]
  Name:            KakaoTalk_20260526_174632410_03.jpg
  Dimensions:      2252 x 4000
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    2628.57px (Prior)
  GPS:             LAT=36.103, LON=129.387, ALT=82.000
  Features:        13918
Processed file [2/25]
  Name:            KakaoTalk_20260526_174632410_02.jpg
  Dimensions:      2252 x 4000
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    2628.57px (Prior)
  GPS:             LAT=36.103, LON=129.387, ALT=82.000
  Features:        15460
Processed file [3/25]
  Name:            KakaoTalk_20260526_174632410_01.jpg
  Dimensions:      2252 x 4000
  Camera:          #1 - SIMPLE_RADIAL
  Focal Length:    2628.57px (Prior)
  GPS:             LAT=36.103, LON=129.387, ALT=82.000
  Features:        14594
Processed file [4/25]
  Name:            KakaoTalk_20260526_174632410.jpg
  Dimensions:      2252 x 4000
  Camera:          #1 - SIMPLE_RADIAL
  F

In [ ]:
import pycolmap
print(pycolmap.__version__)
print(dir(pycolmap.ImageReaderOptions()))

4.0.4
['__class__', '__copy__', '__deepcopy__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '_pybind11_conduit_v1_', 'camera_mask_path', 'camera_model', 'camera_params', 'check', 'default_focal_length_factor', 'existing_camera_id', 'mask_path', 'mergedict', 'summary', 'todict']


---
## CELL 5 | SfM — Feature Matching
> 이미지 수에 따라 매칭 전략 자동 선택

In [ ]:
import os, subprocess

print('Feature Matching 중... (1~3분 소요)\n')

env = os.environ.copy()
env['QT_QPA_PLATFORM'] = 'offscreen'

cmd = [
    'colmap', 'exhaustive_matcher',
    '--database_path', DB,
    '--SiftMatching.use_gpu', '0',
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\n종료 코드: {process.returncode}')
if process.returncode == 0:
    print('✅ Matching 완료!')
else:
    print('ERROR: Matching 실패')

Feature Matching 중... (1~3분 소요)


Exhaustive feature matching

Matching block [1/1, 1/1] in 452.370s
Elapsed time: 7.544 [minutes]

종료 코드: 0
✅ Matching 완료!


---
## CELL 6 | SfM — Sparse Reconstruction

In [ ]:
import os, subprocess, shutil

if os.path.exists(SPARSE):
    shutil.rmtree(SPARSE)
os.makedirs(SPARSE, exist_ok=True)

print('Sparse Reconstruction 중...\n')

env = os.environ.copy()
env['QT_QPA_PLATFORM'] = 'offscreen'

cmd = [
    'colmap', 'mapper',
    '--database_path', DB,
    '--image_path', IMAGES,
    '--output_path', SPARSE,
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\n종료 코드: {process.returncode}')

if process.returncode == 0:
    # 결과 확인
    recons = [d for d in os.listdir(SPARSE) if os.path.isdir(os.path.join(SPARSE, d))]
    print(f'생성된 reconstruction: {recons}')
    for r in recons:
        cameras = os.path.join(SPARSE, r, 'cameras.bin')
        if os.path.exists(cameras):
            print(f'  [{r}] cameras.bin ✅')
else:
    print('ERROR: Sparse reconstruction 실패')

Sparse Reconstruction 중...


Loading database

Loading cameras... 1 in 0.000s
Loading matches... 300 in 0.011s
Loading images... 25 in 0.027s (connected 25)
Building correspondence graph... in 0.063s (ignored 0)

Elapsed time: 0.002 [minutes]


Finding good initial image pair


Initializing with image pair #16 and #18


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  2.251540e+03    0.00e+00    7.37e+04   0.00e+00   0.00e+00  1.00e+04        0    5.16e-03    1.39e-02
   1  1.699601e+03    5.52e+02    3.37e+04   6.44e+00   1.01e+00  3.00e+04        1    1.31e-02    2.71e-02
   2  1.680643e+03    1.90e+01    2.55e+04   1.15e+00   1.07e+00  9.00e+04        1    9.46e-03    3.66e-02
   3  1.721368e+03   -4.07e+01    2.55e+04   1.43e+01  -3.60e+00  4.50e+04        1    4.81e-03    4.15e-02
   4  1.678247e+03    2.40e+00    1.03e+05   8.55e+00   3.04e-01  4.25e+04        1    9.23e-03    5.07e-02
   5

---
## CELL 7 | MVS — Dense Reconstruction

In [ ]:
# COLMAP GitHub 릴리즈 목록 확인
!pip install requests -q
import requests

resp = requests.get('https://api.github.com/repos/colmap/colmap/releases/latest')
data = resp.json()
print(f'최신 버전: {data["tag_name"]}')
print('\n다운로드 가능한 파일들:')
for asset in data['assets']:
    print(f'  {asset["name"]}')
    print(f'  {asset["browser_download_url"]}')

최신 버전: 4.0.4

다운로드 가능한 파일들:
  colmap-x64-windows-cuda.zip
  https://github.com/colmap/colmap/releases/download/4.0.4/colmap-x64-windows-cuda.zip
  colmap-x64-windows-nocuda.zip
  https://github.com/colmap/colmap/releases/download/4.0.4/colmap-x64-windows-nocuda.zip


In [ ]:
import torch, os, subprocess

PROJECT_NAME = 'recon3d'
BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
IMAGES = f'{BASE}/images'
SPARSE = f'{BASE}/sparse'
DENSE  = f'{BASE}/dense'
OUTPUT = f'{BASE}/output'

env = os.environ.copy()
env['QT_QPA_PLATFORM'] = 'offscreen'

vram_gb    = torch.cuda.get_device_properties(0).total_memory / 1024**3
cache_size = max(8, int(vram_gb * 0.5))
print(f'VRAM: {vram_gb:.1f}GB → cache_size: {cache_size}GB')

sparse_path = os.path.join(SPARSE, '0')

def run(cmd, label):
    print(f'\n[{label}] 시작...')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    print(f'종료 코드: {p.returncode}')
    return p.returncode

run([
    'colmap', 'image_undistorter',
    '--image_path', IMAGES,
    '--input_path', sparse_path,
    '--output_path', DENSE,
    '--output_type', 'COLMAP',
    '--max_image_size', '1600',
], '1/3 Undistortion')

run([
    'colmap', 'patch_match_stereo',
    '--workspace_path', DENSE,
    '--workspace_format', 'COLMAP',
    '--PatchMatchStereo.geom_consistency', 'true',
    '--PatchMatchStereo.max_image_size', '1600',
    '--PatchMatchStereo.cache_size', str(cache_size),
    '--PatchMatchStereo.gpu_index', '0',
], '2/3 Dense Matching (GPU)')

fused = f'{DENSE}/fused.ply'
run([
    'colmap', 'stereo_fusion',
    '--workspace_path', DENSE,
    '--workspace_format', 'COLMAP',
    '--input_type', 'geometric',
    '--output_path', fused,
], '3/3 Stereo Fusion')

if os.path.exists(fused):
    mb = os.path.getsize(fused) / 1024**2
    print(f'\n✅ MVS 완료! fused.ply: {mb:.1f} MB')
else:
    print('\nERROR: fused.ply 생성 실패')

VRAM: 14.6GB → cache_size: 8GB

[1/3 Undistortion] 시작...

Reading reconstruction

 => Reconstruction with 25 images and 23758 points

Image undistortion

Undistorting image [1/25]
Undistorting image [2/25]
Undistorting image [3/25]
Undistorting image [4/25]
Undistorting image [5/25]
Undistorting image [6/25]
Undistorting image [7/25]
Undistorting image [8/25]
Undistorting image [9/25]
Undistorting image [10/25]
Undistorting image [11/25]
Undistorting image [12/25]
Undistorting image [13/25]
Undistorting image [14/25]
Undistorting image [15/25]
Undistorting image [16/25]
Undistorting image [17/25]
Undistorting image [18/25]
Undistorting image [19/25]
Undistorting image [20/25]
Undistorting image [21/25]
Undistorting image [22/25]
Undistorting image [23/25]
Undistorting image [24/25]
Undistorting image [25/25]
Writing reconstruction...
Writing configuration...
Writing scripts...
Elapsed time: 0.628 [minutes]
종료 코드: 0

[2/3 Dense Matching (GPU)] 시작...
ERROR: Dense stereo reconstruction re

---
## CELL 8 | 3DGS 환경 설치 (CUDA 버전 자동 감지)

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'사용가능: {torch.cuda.is_available()}')

!pip install plyfile tqdm pillow -q
print('설치 완료')

PyTorch: 2.10.0+cu128
CUDA: 12.8
사용가능: True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.4 MB/s eta 0:00:00
설치 완료


In [ ]:
import torch

cuda_ver = torch.version.cuda
print(f'감지된 CUDA: {cuda_ver}')

if cuda_ver.startswith('11.8'):
    !pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 \
        --index-url https://download.pytorch.org/whl/cu118 -q
elif cuda_ver.startswith('12.1'):
    !pip install torch==2.1.0+cu121 torchvision==0.16.0+cu121 \
        --index-url https://download.pytorch.org/whl/cu121 -q
elif cuda_ver.startswith('12.2') or cuda_ver.startswith('12.3') or cuda_ver.startswith('12.4'):
    !pip install torch==2.1.0+cu121 torchvision==0.16.0+cu121 \
        --index-url https://download.pytorch.org/whl/cu121 -q
else:
    print(f'CUDA {cuda_ver} - 기본 PyTorch 유지')

!pip install plyfile tqdm pillow -q

import importlib; importlib.invalidate_caches()
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'사용가능: {torch.cuda.is_available()}')

---
## CELL 9 | 3DGS 다운로드 및 빌드 (3~5분 소요)

In [ ]:
%%bash
cd /content

if [ -d 'gaussian-splatting' ]; then
    echo '이미 존재, 스킵'
    exit 0
fi

git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive --quiet
echo '클론 완료'

cd gaussian-splatting
echo 'diff-gaussian-rasterization 빌드 중... (3~5분)'
pip install submodules/diff-gaussian-rasterization -q
echo 'simple-knn 빌드 중...'
pip install submodules/simple-knn -q
echo '빌드 완료!'

클론 완료
diff-gaussian-rasterization 빌드 중... (3~5분)
simple-knn 빌드 중...
빌드 완료!


---
## CELL 10 | 3DGS 학습

In [ ]:
import os, subprocess, shutil

PROJECT_NAME = 'recon3d'
BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
IMAGES = f'{BASE}/images'
SPARSE = f'{BASE}/sparse'
DB     = f'{BASE}/database.db'

# 기존 결과 삭제
for path in [DB, f'{DB}-wal', f'{DB}-shm']:
    if os.path.exists(path): os.remove(path)
if os.path.exists(SPARSE): shutil.rmtree(SPARSE)
os.makedirs(SPARSE, exist_ok=True)
print('기존 결과 삭제 완료')

# hloc 설치
print('\nhloc 설치 중...')
!pip install git+https://github.com/cvg/Hierarchical-Localization.git -q
print('hloc 설치 완료')

기존 결과 삭제 완료

hloc 설치 중...
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 64.8 MB/s eta 0:00:00
hloc 설치 완료


In [ ]:
# SuperGluePretrainedNetwork 서브모듈 수동 설치
!pip uninstall hloc -y -q
!pip install git+https://github.com/cvg/Hierarchical-Localization.git@master -q

import subprocess
# SuperGlue 모델 직접 클론
!git clone https://github.com/magicleap/SuperGluePretrainedNetwork.git \
    /usr/local/lib/python3.12/dist-packages/hloc/extractors/SuperGluePretrainedNetwork --quiet

print('완료')

  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
완료


In [ ]:
# 다시 시도
import sys
sys.path.append('/usr/local/lib/python3.12/dist-packages/hloc/extractors')

from pathlib import Path
from hloc import extract_features, match_features, reconstruction

IMAGE_DIR  = Path(IMAGES)
OUTPUT_DIR = Path(BASE) / 'hloc_output'
OUTPUT_DIR.mkdir(exist_ok=True)
SPARSE_DIR = Path(SPARSE)

print('SuperPoint Feature Extraction 중... (GPU)')
feature_conf = extract_features.confs['superpoint_aachen']
feature_path = extract_features.main(
    feature_conf,
    IMAGE_DIR,
    OUTPUT_DIR
)
print(f'완료: {feature_path}')

[2026/05/26 15:40:35 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-r1024',
 'preprocessing': {'grayscale': True, 'resize_max': 1024}}
[2026/05/26 15:40:35 hloc INFO] Found 25 images in root /content/drive/MyDrive/recon3d/images.


SuperPoint Feature Extraction 중... (GPU)
Loaded SuperPoint model


100%|██████████| 25/25 [00:06<00:00,  3.96it/s]
[2026/05/26 15:40:41 hloc INFO] Finished exporting features.


완료: /content/drive/MyDrive/recon3d/hloc_output/feats-superpoint-n4096-r1024.h5


In [ ]:
from hloc import reconstruction
from pathlib import Path

IMAGE_DIR  = Path(IMAGES)
OUTPUT_DIR = Path(BASE) / 'hloc_output'
SPARSE_DIR = Path(SPARSE)

# reconstruction.main 파라미터 확인
import inspect
print(inspect.signature(reconstruction.main))

(sfm_dir: pathlib.Path, image_dir: pathlib.Path, pairs: pathlib.Path, features: pathlib.Path, matches: pathlib.Path, camera_mode: pycolmap._core.CameraMode = CameraMode.AUTO, verbose: bool = False, skip_geometric_verification: bool = False, min_match_score: Optional[float] = None, image_list: Optional[List[str]] = None, image_options: Optional[Dict[str, Any]] = None, mapper_options: Optional[Dict[str, Any]] = None) -> pycolmap._core.Reconstruction


In [ ]:
from hloc import reconstruction
from pathlib import Path
import pycolmap

IMAGE_DIR  = Path(IMAGES)
OUTPUT_DIR = Path(BASE) / 'hloc_output'
SPARSE_DIR = Path(SPARSE)

print('Sparse Reconstruction 중...')
model = reconstruction.main(
    sfm_dir=SPARSE_DIR,
    image_dir=IMAGE_DIR,
    pairs=OUTPUT_DIR / 'pairs-exhaustive.txt',
    features=OUTPUT_DIR / 'feats-superpoint-n4096-r1024.h5',
    matches=OUTPUT_DIR / 'feats-superpoint-n4096-r1024_matches-superglue_pairs-exhaustive.h5',
    camera_mode=pycolmap.CameraMode.SINGLE,
    image_options={'camera_model': 'PINHOLE'},  # 3DGS 호환
    verbose=True,
)

print(f'\n✅ SfM 완료!')
print(f'   등록 이미지: {model.num_reg_images()} / 25')
print(f'   3D 포인트:   {model.num_points3D():,}개')

[2026/05/26 15:43:31 hloc INFO] Writing COLMAP logs to /content/drive/MyDrive/recon3d/sparse/colmap.LOG.*
[2026/05/26 15:43:31 hloc INFO] Creating an empty database...
[2026/05/26 15:43:31 hloc INFO] Importing images into the database...


Sparse Reconstruction 중...


[2026/05/26 15:43:38 hloc INFO] Importing features into the database...
100%|██████████| 25/25 [00:00<00:00, 213.20it/s]
[2026/05/26 15:43:38 hloc INFO] Importing matches into the database...
100%|██████████| 300/300 [00:06<00:00, 44.24it/s] 
[2026/05/26 15:43:45 hloc INFO] Performing geometric verification of the matches...
[2026/05/26 15:44:41 hloc INFO] Running 3D reconstruction...
Reconstruction 0: 100%|██████████| 25/25 [00:21<00:00,  1.18images/s, registered]
[2026/05/26 15:45:02 hloc INFO] Reconstructed 1 model(s).
[2026/05/26 15:45:02 hloc INFO] Largest model is #0 with 25 images.
[2026/05/26 15:45:02 hloc INFO] Reconstruction statistics:
Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 25
	num_reg_frames = 25
	num_images = 25
	num_points3D = 4880
	num_observations = 14956
	mean_track_length = 3.06475
	mean_observations_per_image = 598.24
	mean_reprojection_error = 1.39446
	num_input_images = 25



✅ SfM 완료!
   등록 이미지: 25 / 25
   3D 포인트:   4,880개


In [ ]:
import os

# sparse 폴더 구조 확인
for root, dirs, files in os.walk(SPARSE):
    level = root.replace(SPARSE, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

sparse/
  database.db
  colmap.LOG.20260526-154345.10327
  rigs.bin
  cameras.bin
  frames.bin
  images.bin
  points3D.bin
  models/
    0/


In [ ]:
import os, shutil

sparse_0 = os.path.join(SPARSE, '0')
os.makedirs(sparse_0, exist_ok=True)

# 필요한 파일 3개를 sparse/0/ 로 복사
for fname in ['cameras.bin', 'images.bin', 'points3D.bin']:
    src = os.path.join(SPARSE, fname)
    dst = os.path.join(sparse_0, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'복사: {fname} ✅')
    else:
        print(f'없음: {fname} ❌')

print('\n구조 확인:')
for f in os.listdir(sparse_0):
    print(f'  sparse/0/{f}')

복사: cameras.bin ✅
복사: images.bin ✅
복사: points3D.bin ✅

구조 확인:
  sparse/0/cameras.bin
  sparse/0/images.bin
  sparse/0/points3D.bin


In [ ]:
import os, subprocess

PROJECT_NAME = 'recon3d'
BASE   = f'/content/drive/MyDrive/{PROJECT_NAME}'
GS_OUT = f'{BASE}/output/gs_result'

!pip install plyfile tqdm pillow -q

env = os.environ.copy()
env['QT_QPA_PLATFORM'] = 'offscreen'

print('3DGS 학습 재시작 (6000부터)...\n')

p = subprocess.Popen(
    ['python', 'train.py',
     '-s', BASE,
     '-m', GS_OUT,
     '--iterations', '30000',
     '--save_iterations', '6000', '9000', '12000', '15000', '20000', '30000',
     '--checkpoint_iterations', '6000', '9000', '12000', '15000', '20000', '30000',  # ← .pth 저장 추가
     '--test_iterations', '30000',
     '--densify_until_iter', '5000',   # 5000으로 낮춤 (메모리 안정)
     '--densify_grad_threshold', '0.0002',
    ],
    cwd='/content/gaussian-splatting',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env
)
for line in p.stdout:
    print(line, end='', flush=True)
p.wait()

print('\n=== 저장된 체크포인트 ===')
pc_dir = f'{GS_OUT}/point_cloud'
if os.path.exists(pc_dir):
    for d in sorted(os.listdir(pc_dir)):
        ply = f'{pc_dir}/{d}/point_cloud.ply'
        if os.path.exists(ply):
            mb = os.path.getsize(ply) / 1024**2
            print(f'  ✅ {d}: {mb:.1f} MB')

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
Training progress:  99%|█████████▉| 29770/30000 [1:41:38<00:48,  4.73it/s, Loss=0.0886364, Depth Loss=0.0000000]
